In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import hyperbolic as hb

In [ ]:
import math
import numpy as np

import drawSvg as draw
from drawSvg import Drawing
import hyperbolic
from hyperbolic import euclid, util
from hyperbolic.euclid.shapes import Circle as ECircle
from hyperbolic.poincare.shapes import *
from hyperbolic.poincare import Transform
from hyperbolic.poincare.util import radialEuclidToPoincare, radialPoincareToEuclid, \
                                     poincareToEuclidFactor, triangleSideForAngles
import hyperbolic.tiles as htiles

class Circle(Circle):
    def __init__(self, projShape, center=None, r=None):
        super().__init__(projShape)
        if not isinstance(projShape, ECircle):
            raise ValueError('projShape must be a euclidean circle')
        self.projShape = projShape
        if r is None or center is None:
            de0 = math.hypot(projShape.cx, projShape.cy)
            de1 = de0 - projShape.r
            de2 = de0 + projShape.r
            dh1 = radialEuclidToPoincare(de1)
            dh2 = radialEuclidToPoincare(de2)
        if r is None:
            r = (dh2 - dh1) / 2
        if center is None:
            cr = (dh2 + dh1) / 2
            theta = math.atan2(projShape.cx, projShape.cy)
            center = Point.fromHPolar(cr, theta)
        self.r = r
        self.center = center

In [ ]:
def drawTiles(drawing, tiles):
    for tile in tiles:
        global t
        t = tile
        d.draw(tile, hwidth=0.02, fill='white')
    for tile in tiles:
        d.draw(tile, drawVerts=True, hradius=0.15, hwidth=0.02,
                     fill='black', opacity=0.6)

In [ ]:
class GraphTileLayout(htiles.TileLayout):
    def tilePlane(self, startTile, depth=2, return_edges=False):
        edges = set()
        tiles = [startTile]
        boundary = [(side, startTile) for side in startTile.sides]
        for j in range(depth):
            boundary2 = []
            i = 0
            while i < len(boundary):
                tile = self.placeTile(boundary[i][0])
                tiles.append(tile)
                edges.add((tile, boundary[i][1]))
                sides = tile.permutedSides()
                o = 1
                p = len(sides)
                if i == 0:
                    if sides[o] == boundary[-1][0]:
                        o += 1
                        edges.add((tile, boundary[-1][1]))
                        boundary.pop()
                else:
                    if sides[o] == boundary2[-1][0]:
                        o += 1
                        edges.add((tile, boundary2[-1][1]))
                        boundary2.pop()
                    if sides[p-1] == boundary[(i + 1) % len(boundary)][0]:
                        p -= 1
                        edges.add((tile, boundary[(i + 1) % len(boundary)][1]))
                        i += 1
                    if sides[p-1] == boundary2[0][0]:
                        p -= 1
                        edges.add((tile, boundary2[0][1]))
                        boundary2.pop(0)
                boundary2.extend([[side, tile] for side in sides[o:p]])
                i += 1
            boundary = boundary2
        return (tiles, edges) if return_edges else tiles

In [ ]:
from itertools import chain
#def construct_circumcircle(polygon):
def get_circumcenter(polygon):
    if isinstance(polygon, hyperbolic.tiles.Tile):
        polygon = polygon.toPolygon()
    
    c = ECircle.fromPoints(*chain(*[[v.x, v.y] for v in polygon.vertices[:3]]))
    #return c.cx, c.cy
    c = Circle(c)
    #print(c.center.x, c.center.y, c.r)
    return c.center.x, c.center.y
#return vs[0]
#v = construct_circumcircle(tiles[5])
#type(vs[0].__dict__)

In [ ]:
import eucare as ec
import networkx as nx

plotting_kwargs = {
    'figsize': (5, 5),
    'render_faces': False,
    'render_vertices': False,
    'render_edges': True,
    'face_inset': 0,
    'line_width': 3,
}
render_settings = plotting_kwargs

In [ ]:
# Control the orientation that tiles are placed together
class TileLayoutIsosceles(GraphTileLayout):
    def calcGenIndex(self, code):
        ''' Controls which type of tile to place '''
        return 0
    def calcTileTouchSide(self, code, genIndex):
        ''' Controls tile orientation '''
        try:
            side, colors = code
            return 2 - side
        except TypeError:
            return 0
    def calcSideCodes(self, code, genIndex, touchSide, defaultCodes):
        ''' Controls tile side codes '''
        try:
            side, colors = code
            # 0=red, 1=orange, 2=yellow, 3=lime, 4=green, 5=blue, 6=pink
            if side != 1:
                if side == 0: shift = -1
                elif side == 2: shift = 1
                else: shift = 0
                nc = len(colors)
                newColors = [colors[(i+shift)%nc] for i in range(nc)]
            else:
                newColors = [colors[0], colors[1], colors[6], colors[4],
                    colors[3], colors[5], colors[2]]
        except TypeError:
            nc = q1
            newColors = [(i+code)%nc for i in range(nc)]
        return [(side, newColors) for side in range(3)]
    
q1 = 7  # Number of polygons around some points
q2 = 6  # Number of polygons around other points
depth = 14  # How far from the center to draw tiles

# Calculate isosceles triangle
assert q2 > 4 and q2 % 2 == 0, 'q2 must be even and at least 6'
phi1, phi2 = math.pi*2/q1, math.pi*2/q2
# Side lengths
s0 = triangleSideForAngles(phi1, phi2, phi2)
s1 = triangleSideForAngles(phi2, phi2, phi1)
s2 = s0
pt0 = Point.fromHPolar(0,0)
pt1 = Point.fromHPolar(s0,0)
pt2 = Point.fromHPolar(s2,phi1)
# Circumcircle
circumcirc = euclid.shapes.Circle.fromPoints(*pt0, *pt1, *pt2)
r = radialEuclidToPoincare(circumcirc.r)
ptCenter = Point.fromEuclid(circumcirc.cx, circumcirc.cy)
# Translate triangle to center
transCenter = Transform.shiftOrigin(ptCenter, pt0)
ptc0, ptc1, ptc2 = transCenter(pt0, pt1, pt2)
centerPoints = (ptc0, ptc1, ptc2)
tile = htiles.Tile(centerPoints)


# Calculate weave width
# For right triangle: tan(A) = tanh(opp) / sinh(adj)
# => opp = atanh(tan(A) * sinh(adj))
rInsc = math.atanh(math.tan(phi2/2) * math.sinh(s1/2))  # Inscribed circle radius
h = math.atanh(math.tan(phi2) * math.sinh(s1/2))  # Triangle height
centerDiff = r - (h - rInsc)

tGen = htiles.TileGen.fromCenterTile(tile)

decoratorLate = htiles.TileDecoratorLateInit()

tLayout = TileLayoutIsosceles()
tLayout.addGenerator(tGen, (0,)*4, decoratorLate)
startTile = tLayout.startTile(code=2,rotateDeg=0,centerCorner=False)

tiles, edges = tLayout.tilePlane(startTile, depth=depth, return_edges=True)

d = draw.Drawing(2, 2, origin='center')
d.draw(euclid.shapes.Circle(0, 0, 1), fill='#ddd')
drawTiles(d, tiles)

d.setRenderSize(w=400)
d.saveSvg('isosceles-{}-{}.svg'.format(q1, q2))
d

In [ ]:
# Regular tesselation
p = 7
q = 3
depth = 5

theta = math.pi*2/p
phi = math.pi*2/q
r = triangleSideForAngles(theta/2, phi, theta/2)

tGen = htiles.TileGen.makeRegular(p, hr=r, skip=1)

tLayout = GraphTileLayout()
tLayout.addGenerator(tGen, (0,)*p)

startTile = tLayout.defaultStartTile(rotateDeg=90)

tiles, edges = tLayout.tilePlane(startTile, depth=depth, return_edges=True)

d = Drawing(2, 2, origin='center')
d.draw(euclid.shapes.Circle(0, 0, 1), fill='silver')
drawTiles(d, tiles)

d.setRenderSize(w=400)
d.saveSvg('tileTriangleSquare.svg')
d

In [ ]:
G = nx.Graph()
circumcenters = {tile: get_circumcenter(tile) for tile in tiles}
#print(circumcenters.values())

G.add_edges_from([(circumcenters[t1], circumcenters[t2])
                 for t1, t2 in edges])

plt.figure()
nx.draw(G, {n: np.array(n) for n in G.nodes})#, list(G.nodes))
plt.gca().set_aspect('equal')
plt.show()
# assert False
G = ec.conversions.EHEG_from_nx(G)
G.show(**plotting_kwargs)

ps = G.get_position_view(return_vertices=False)
zs = np.array([complex(*vals) for vals in ps])

#zs = np.arctanh(zs)

#directions = np.exp(1j * np.linspace(0, 2*np.pi, 3, endpoint=False))
#zs = np.sum([d.conj()*np.log(1-d*zs) for d in directions], axis=0)

ps[:] = np.stack([zs.real, zs.imag], axis=-1)


#G = ec.conway.join_graph()(G)

G.show(**plotting_kwargs)

In [ ]:

from eucare.base import signed_area

for e in G.halfedges:
    if THIS_WAY in e:
        del e[THIS_WAY]

def pointing_away(e):
    return (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()

# for normal
# eps = 1e-8
# for e in G.halfedges:
#     radial_difference = (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()
#     if radial_difference > eps or (abs(radial_difference) < eps and signed_area(np.array([[0, 0], e.orig['pos'], e.dest['pos']])) < 0):
#         e[THIS_WAY] = True
#         assert THIS_WAY not in e.rev
        
        
vertices = list(G.vertices)
central_vertex = vertices[np.argmin([np.linalg.norm(v['pos']) for v in vertices])]
hops_to_central = {central_vertex: 0}
boundary = {central_vertex}
i = 1
while boundary:
    next_boundary = set()
    for v in boundary:
        for v2 in v.vertex_iter():
            if v2 not in hops_to_central and v2:
                next_boundary.add(v2)
                hops_to_central[v2] = i
    i += 1
    boundary = next_boundary
    
eps = 1e-8
for e in G.halfedges:
    radial_difference = (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()
    if radial_difference > eps or (abs(radial_difference) < eps and signed_area(np.array([[0, 0], e.orig['pos'], e.dest['pos']])) < 0):
        if hops_to_central[e.orig] % 2 == 0:
            e = e.rev
        e[THIS_WAY] = True
        assert THIS_WAY not in e.rev

## wrong
# def squared_edge_length(e):
#     return ((e.orig['pos'] - e.dest['pos'])**2).sum()

# verts = G.vertices
# verts = sorted(verts, key=lambda v: -max(*[squared_edge_length(e) for e in v.outgoing_iter()]))
# for v in verts:
#     for e in v.outgoing_iter():
#         e[THIS_WAY] = True
#         if THIS_WAY in e.rev:
#             del e.rev[THIS_WAY]

# for join
# for f in G.faces:
#     if f.order() == 4:
#         for e in f.halfedge_iter():
#             if e.orig.order() == 7:
#                 e[THIS_WAY] = True                

In [ ]:
from eucare.overlap import THIS_WAY, assign_shrink_rotate_creases

def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

SRG = ec.reciprocal_figures.shrink_rotate_graph(G)
SRG.recompute_lengths_and_angles()
assign_shrink_rotate_creases(SRG)

colors = {
    0: (0, 0, 0),
    1: (1, 0, 0),
    -1: (0, 0, 1)
}
for e in SRG.halfedges:
    e['color_key'] = colors[e.attributes.get('crease_assignment', 0)]
# join unneccessary boundary vertices
to_join = []
for v in SRG.vertices:
    if v.on_border() and v.order() == 2:
        to_join.append(v)
for v in to_join:
    SRG.join_vertex(v)
SRG.recompute_lengths_and_angles()

SRG.show(**plotting_kwargs)

mks = max_kawasaki_sum(SRG)
print(mks)

In [ ]:
%matplotlib notebook
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection
from eucare.plotting import set_equal_aspect
from eucare.reciprocal_figures import random_directed_set

def reshrinkrotate(alpha, factor, global_scale=1):
    for f in faces:
        if 'rotation_center' not in f.attributes:
            continue
        ps, vs = np.array([[v['base_pos'], v] for v in f.vertex_iter()]).T
        ps = np.stack(ps)
        rotation_center = f['rotation_center']

        ps = rotation_center + (ps - rotation_center) @ ec.base.rotation_matrix(alpha) * factor
        
        if global_scale != 1:
            ps *= global_scale
            
        for v, p in zip(vs, ps):
            v['pos'] = p
            
def get_segments(edges):
    return np.array([[e.orig['pos'], e.dest['pos']] for  e in edges])

def get_polys(faces):
    return [[v['pos'] for v in f.vertex_iter()] for f in faces]

edges = list(random_directed_set(SRG.halfedges))
faces = list(SRG.faces)

#%timeit SRG.show(**render_settings)
#%timeit reshrinkrotate(np.pi/9, 0.7)
#%timeit get_segments(edges)
#%timeit get_polys(faces)

segments = get_segments(edges)
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1, 1, 1)
lc = LineCollection(segments, antialiased=True, color='k', linewidth=1)

pc = PolyCollection(get_polys(faces), antialiased=True, color='k')
pc.set_alpha(0.1)

polys = ax.add_collection(pc)
lines = ax.add_collection(lc)

ax.autoscale()
set_equal_aspect()
plt.draw()

alpha_slider = widgets.FloatSlider(0.166666, min=-1, max=1, step=0.02)
factor_slider = widgets.FloatSlider(0.58, min=0, max=6, step=0.05)

last_reparametrized = False
def update(alpha, factor, folded=False, reparametrized=False, scale_folded=False, show_lines=False, show_polys=True):
    alpha = alpha * np.pi
    global last_reparametrized
    if not last_reparametrized:
        gamma = factor / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1)
        beta = np.arccos(np.sin(alpha) / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1))
    else:
        gamma = factor
        beta = alpha
        # TODO: sign
        alpha = np.arccos((gamma + np.sin(beta)) / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1))
        factor = gamma / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1)
    
    if reparametrized is not last_reparametrized:
        # adjust slider values
        last_reparametrized = reparametrized
        if reparametrized:
            alpha_slider.value = beta / np.pi
            factor_slider.value = gamma
        else:
            alpha_slider.value = alpha / np.pi
            factor_slider.value = factor
    
    if not folded:
        reshrinkrotate(alpha, factor)
    else:
        factor_folded = gamma / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1)
        alpha_folded = np.sign(alpha) * np.arccos((gamma - np.sin(beta)) / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1))
        reshrinkrotate(alpha_folded, factor_folded, 
                       global_scale=1 if not scale_folded else factor/factor_folded)
    lines.set_segments(get_segments(edges) if show_lines else [])
    polys.set_paths(get_polys(faces) if show_polys else [])
    #ax.draw_artist(lc)
    fig.canvas.draw_idle()
    #print(f'gamma {gamma}, beta {beta * 360 / (2 * np.pi)}')

widgets.interact(
    update,
    alpha=alpha_slider,
    factor=factor_slider,
    
);

In [ ]:
from scipy.optimize import minimize_scalar, basinhopping
from copy import copy

def angle_to_height(G, angle):
    border_positions = np.array([v['pos'] for v in G.border_vertex_iter()])
    rot_border_positions = border_positions @ np.array([[np.cos(angle)], [-np.sin(angle)]])
    return np.max(rot_border_positions) - np.min(rot_border_positions)

def optimize_rotation(G):
    border_positions = np.array([v['pos'] for v in G.border_vertex_iter()])

    def angle_to_height(angle):
#         if isinstance(angle, np.ndarray):
#             angle = angle[0]
        rot_border_positions = border_positions @ np.array([[np.cos(angle)], [-np.sin(angle)]])
        return np.max(rot_border_positions) - np.min(rot_border_positions)
    
    #minimizer_kwargs = dict(method="L-BFGS-B", bounds=[[0, 2*np.pi]])
    #result = basinhopping(angle_to_height, x0=0, stepsize=np.pi/100, minimizer_kwargs=minimizer_kwargs)
    #angle = result['x'][0]
    
    angles = np.linspace(0, np.pi, 10000)
    heights = [angle_to_height(a) for a in angles]
    plt.figure()
    plt.plot(angles, heights)
    plt.show()
    angle = angles[np.argmin(heights)]
    print(angle)
    ps = G.get_position_view(return_vertices=False)
    ps[:] = ps @ np.array([[np.cos(angle), np.sin(angle)], [-np.sin(angle), np.cos(angle)]])
    
def min_edge_length(G):
    edges = copy(G.halfedges)
    min_length = np.inf
    while edges:
        e = edges.pop()
        edges.remove(e.rev)
        min_length = min(((e.orig['pos'] - e.dest['pos'])**2).sum(), min_length)
    return np.sqrt(min_length)

min_foldable_length = 0.5 #cm
min_edge_length(SRG)

optimize_rotation(SRG)
sheet_height = angle_to_height(SRG, 0) * min_foldable_length / min_edge_length(SRG)
print('sheet heigth:', sheet_height, 'cm')

# angles = np.linspace(0, np.pi/2, 100)
# heights = [angle_to_height(SRG, a) for a in angles]
# angles[np.argmin(heights)]
# plt.figure()
# plt.plot(angles, heights)
# plt.show();

In [ ]:
from eucare.overlap import fold_complete
SRG.recompute_lengths_and_angles()
result = fold_complete(SRG.copy(), overlap_eps=1e-7, area_eps=0)
render_settings['render_faces'] = False
result['CP'].show(**render_settings)
result['folded_view_top'].show(**render_settings)
result['folded_view_bottom'].show(**render_settings)

In [ ]:
import os
from eucare.redering import SvgwriteRenderer
from eucare.overlap import save_results

path = 'nice_images/hyperbolic_7-3_huge_f1.5'
save_results(result, path, render_settings)
plotter = SvgwriteRenderer()
plotter.render_graph(os.path.join(path, 'cp_for_cutting.svg'), result['CP'], height=sheet_height)

In [ ]:
p1 = 5
p2 = 4
q = 2

theta1, theta2 = math.pi*2/p1, math.pi*2/p2
phiSum = math.pi*2/q
r1 = triangleSideForAngles(theta1/2, phiSum/2, theta2/2)
r2 = triangleSideForAngles(theta2/2, phiSum/2, theta1/2)

tGen1 = htiles.TileGen.makeRegular(p1, hr=r1, skip=1)
tGen2 = htiles.TileGen.makeRegular(p2, hr=r2, skip=1)

tLayout = htiles.TileLayout()
tLayout.addGenerator(tGen1, (1,)*p1)
tLayout.addGenerator(tGen2, (0,)*p2)
startTile = tLayout.defaultStartTile(rotateDeg=90)

tiles = tLayout.tilePlane(startTile, depth=2)

d = Drawing(2, 2, origin='center')
d.draw(euclid.shapes.Circle(0, 0, 1), fill='silver')
drawTiles(d, tiles)

d.setRenderSize(w=400)
d.saveSvg('tileTriangleSquare.svg')
d

In [ ]:
t.__dict__

In [ ]:
type(e.p1)